# pm4py-ucm — scenario synthesis from event logs

This notebook walks through the **scenario-synthesis** pipeline that
turns an event log into an *executable* UCM. The output is a
jUCMNav `.jucm` file with:

* a `variant_id` enumeration variable on the URN spec,
* one `ScenarioDef` per concurrency-aware variant cluster, each
  initialising `variant_id` to its own ID,
* `variant_id == "v_i"` disjunction conditions on the OR-fork
  outgoing arcs that drive the jUCMNav traversal engine to pick
  the right branch for each scenario,
* a description on every scenario carrying its partial-order
  expression and the list of original case IDs it covers.

## What's novel here

Two log traces that differ only by the interleaving order of
activities inside a parallel block share the same *choice signature*
and therefore the same variant. So `X → (Y ∥ Z) → W` traces
`X-Y-Z-W` and `X-Z-Y-W` cluster as one. Sequence-variant analysis
splits them.

We also report a **fitness percentage** — the fraction of cases that
replayed cleanly on the discovered tree. Traces that don't fit go
into a `v_noise` bucket and get no scenario emitted, which is the
honest answer when the model is an approximation.

## 1. Setup

Requires `pm4py-ucm` installed (`pip install -e .` from the repo
root). The `graphviz` binary is only needed if you also want to
render the UCM as a PNG; the scenario-synthesis pipeline itself
doesn't depend on it.

In [ ]:
import sys
from pathlib import Path
import shutil

# Make sure we import the WORKTREE pm4py-ucm (the one with the
# scenario-synthesis modules), not any older copy that might be
# pip-installed system-wide. We prepend the repo root to sys.path so
# a fresh notebook works without requiring `pip install -e .` from
# the repo root first.
REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "pm4py_ucm").is_dir():
    if (REPO_ROOT.parent / "pm4py_ucm").is_dir():
        REPO_ROOT = REPO_ROOT.parent  # running from demo/
sys.path.insert(0, str(REPO_ROOT))

import pm4py_ucm
from pm4py_ucm.algo.discovery.variants import clustering as _clustering
from pm4py_ucm.algo.discovery.scenarios import synthesis as _scenarios
from pm4py_ucm.algo.discovery.scenarios import reports as _reports

OUT = Path("scenario_output")
if OUT.exists():
    shutil.rmtree(OUT)
OUT.mkdir()
print(f"pm4py-ucm   {pm4py_ucm.__version__}")
print(f"loaded from {Path(pm4py_ucm.__file__).parent}")

## 2. Synthetic tree — the cleanest demonstration

Tree: `X → (Y ∥ Z) → (A × B) → W`.

Log: 100 cases over three observed sequences:

| Sequence            | Cases |
|---------------------|-------|
| `X-Y-Z-A-W`         |  60   |
| `X-Y-Z-B-W`         |  30   |
| `X-Z-Y-A-W`         |  10   |

Concurrency-aware clustering should produce **2 variants** — the
third sequence is interleaving-equivalent to the first. Naive
sequence-variant analysis would produce 3.

In [ ]:
class T:
    """Duck-typed process tree node — matches pm4py's ProcessTree."""
    def __init__(self, operator=None, label=None, children=None):
        self.operator = operator
        self.label = label
        self.children = children or []

tree = T(operator="->", children=[
    T(label="X"),
    T(operator="+", children=[T(label="Y"), T(label="Z")]),
    T(operator="X", children=[T(label="A"), T(label="B")]),
    T(label="W"),
])

log = []
for i in range(60):
    log.append((f"caseA_{i}",     ["X", "Y", "Z", "A", "W"]))
for i in range(30):
    log.append((f"caseB_{i}",     ["X", "Y", "Z", "B", "W"]))
for i in range(10):
    log.append((f"caseA_alt_{i}", ["X", "Z", "Y", "A", "W"]))  # parallel re-order

print(f"log: {len(log)} cases")

In [ ]:
result = _clustering.cluster(log, tree)

print(f"Total cases:                 {result.total_cases}")
print(f"Concurrency-aware variants:  {len(result.variants)}")
print(f"Sequence variants in log:    {result.sequence_variant_count}")
print(f"Fitness percentage:          {result.fitness_percentage:.1%}")
print(f"Compression ratio:           {result.compression_ratio:.3f}")
print()
print(f"{'Variant':<10} {'Freq':>6} {'Seqs':>6} {'Linearizations':>15}  Partial-order expression")
print("-" * 80)
for v in result.variants:
    print(
        f"{v.variant_id:<10} {v.frequency:>6} {v.sequence_variants:>6} "
        f"{v.linearization_count:>15}  {v.partial_order_expression}"
    )

### 2.1 Build the UCM and attach scenarios

We could call `pm4py_ucm.discover_scenarios(log)` as a one-shot, but
splitting the calls makes it easier to inspect what's happening at
each step.

In [ ]:
ucm = pm4py_ucm.convert_to_ucm(tree)
group = _scenarios.synthesize_scenarios(ucm, tree, result)

print(f"UCM: {ucm}")
print(f"  enumeration types: {[e.name for e in ucm.enumeration_types]}")
print(f"  variables:         {[v.name for v in ucm.variables]}")
print(f"  scenario groups:   {[g.name for g in ucm.scenario_groups]}")
print()
for sc in group.scenarios:
    inits = ', '.join(f"{i.variable.name}={i.value!r}" for i in sc.initializations)
    print(f"  scenario {sc.name!r}: {inits}")

In [ ]:
jucm_path = OUT / "synthetic.jucm"
pm4py_ucm.write_ucm(ucm, jucm_path)
pm4py_ucm.write_variants_report(result, OUT / "synthetic.scenarios.csv")
pm4py_ucm.write_case_variant_map(result, OUT / "synthetic.case_variant_map.csv")

for p in sorted(OUT.iterdir()):
    print(f"  {p.name}  ({p.stat().st_size:,} bytes)")

### 2.2 Inspect the .jucm scenario layer

The interesting bits — `<scenarioGroups>`, `<variables>`,
`<enumerationTypes>` — all live inside `<ucmspec>`. The OR-fork
outgoing connections carry the `variant_id` conditions inside
`<urndef>` / `<specDiagrams>` / `<connections>`.

In [ ]:
text = jucm_path.read_text(encoding="utf-8")

# Print only the scenario-related lines.
keywords = ("scenarioGroups", "scenarios", "initializations",
            "startPoints", "endPoints",
            "<variables", "<enumerationTypes",
            "variant_id ==")
for line in text.splitlines():
    if any(k in line for k in keywords):
        print(line)

### 2.3 Inspect the CSV reports

In [ ]:
import pandas as pd
variants_df = pd.read_csv(OUT / "synthetic.scenarios.csv")
variants_df

In [ ]:
case_map_df = pd.read_csv(OUT / "synthetic.case_variant_map.csv")
print(f"rows: {len(case_map_df)}")
case_map_df.head(10)

## 3. Loop coarsening — sensitivity check

When a process tree contains a loop, the choice-signature algorithm
coarsens the iteration count to `{0, 1, ≥2}`. Two cases differing
only by iterating 2 vs 5 times collapse into the same variant. Pass
`coarsen_loops=False` to keep each iteration-count distinct.

In [ ]:
loop_tree = T(operator="->", children=[
    T(label="Open"),
    T(operator="*", children=[T(label="Review"), T(label="Revise")]),
    T(label="Close"),
])

loop_log = (
    [(f"once_{i}",  ["Open", "Review", "Close"]) for i in range(50)]
    + [(f"twice_{i}", ["Open", "Review", "Revise", "Review", "Close"]) for i in range(30)]
    + [(f"thrice_{i}", ["Open", "Review", "Revise", "Review", "Revise", "Review", "Close"]) for i in range(20)]
)

coarse = _clustering.cluster(loop_log, loop_tree, coarsen_loops=True)
fine   = _clustering.cluster(loop_log, loop_tree, coarsen_loops=False)
print(f"With coarsening    : {len(coarse.variants)} variants")
for v in coarse.variants:
    print(f"  {v.variant_id}: freq={v.frequency:>3}  {v.partial_order_expression}")
print()
print(f"Without coarsening : {len(fine.variants)} variants")
for v in fine.variants:
    print(f"  {v.variant_id}: freq={v.frequency:>3}  {v.partial_order_expression}")

## 4. Model-fitness loss — what happens to non-conforming cases

Real-world logs almost never conform 100% to the discovered model.
Cases that can't be replayed are bucketed as **noise** — they are
reported in the CSV and the fitness percentage, but no scenario is
synthesized for them (we don't want to fabricate scenarios for
behaviour that the model can't replay).

Here we add a single non-conforming case to the synthetic log.

In [ ]:
noisy_log = list(log) + [
    ("weird_1", ["X", "Y", "Z", "UNKNOWN_ACTIVITY", "W"]),
    ("weird_2", ["X", "A", "W"]),  # skips the (Y || Z) block
]

noisy_result = _clustering.cluster(noisy_log, tree)
print(f"Total cases:           {noisy_result.total_cases}")
print(f"Non-conforming cases:  {len(noisy_result.noise_case_ids)}")
print(f"Fitness percentage:    {noisy_result.fitness_percentage:.2%}")
print(f"Noise case IDs:        {noisy_result.noise_case_ids}")

## 5. Real event log — `IssueTracker.xes`

Same pipeline against one of the bundled sample logs. We use
`pm4py_ucm.discover_scenarios(log)` which runs the full pipeline
(mine tree → build UCM → cluster log → synthesize scenarios) in one
call.

If you run this notebook from inside `demo/`, the bundled XES is at
`../web/samples/IssueTracker.zip` — extract it first if needed.
Otherwise point `LOG_PATH` at any XES file you have handy.

In [ ]:
import zipfile

# Look for the bundled XES log next to this notebook (under demo/)
# and fall back to web/samples/ for users who copied the notebook
# elsewhere.
candidates = [
    Path("IssueTracker.zip"),                       # same dir as notebook
    REPO_ROOT / "demo" / "IssueTracker.zip",
    REPO_ROOT / "web" / "samples" / "IssueTracker.zip",
]
LOG_PATH = Path("IssueTracker.xes")
if not LOG_PATH.exists():
    for zip_path in candidates:
        if zip_path.exists():
            with zipfile.ZipFile(zip_path) as zf:
                zf.extractall(".")
            print(f"extracted {zip_path} -> {LOG_PATH}")
            break
    else:
        print(f"{LOG_PATH} not found in any of:")
        for c in candidates:
            print(f"  - {c}")
        print("Point LOG_PATH at any XES file you have to continue.")
print(f"using log: {LOG_PATH.resolve()}")

In [ ]:
import pm4py
log_df = pm4py.read_xes(str(LOG_PATH))
print(f"events: {len(log_df):,}; cases: {log_df['case:concept:name'].nunique():,}; activities: {log_df['concept:name'].nunique()}")

ucm_real, result_real = pm4py_ucm.discover_scenarios(
    log_df,
    coarsen_loops=True,
    emit_conditions=True,
)
print()
print(f"Variants discovered:  {len(result_real.variants)}")
print(f"Sequence variants:    {result_real.sequence_variant_count}")
print(f"Noise cases:          {len(result_real.noise_case_ids)}")
print(f"Fitness:              {result_real.fitness_percentage:.2%}")
print(f"Compression ratio:    {result_real.compression_ratio:.3f}  (lower = more concurrency collapse)")

In [ ]:
# Top 10 variants by frequency
print(f"{'Variant':<8} {'Freq':>6} {'Seqs':>6} {'Lin':>10}  Partial-order expression")
print("-" * 100)
for v in result_real.variants[:10]:
    print(f"{v.variant_id:<8} {v.frequency:>6} {v.sequence_variants:>6} {v.linearization_count:>10}  {v.partial_order_expression[:80]}")

In [ ]:
real_jucm = OUT / "issue_tracker.jucm"
real_csv  = OUT / "issue_tracker.scenarios.csv"
real_map  = OUT / "issue_tracker.case_variant_map.csv"
pm4py_ucm.write_ucm(ucm_real, real_jucm)
pm4py_ucm.write_variants_report(result_real, real_csv)
pm4py_ucm.write_case_variant_map(result_real, real_map)
for p in (real_jucm, real_csv, real_map):
    print(f"  {p.name}  ({p.stat().st_size:,} bytes)")

## 6. Loading the `.jucm` in jUCMNav

Open `scenario_output/synthetic.jucm` (or `issue_tracker.jucm`) in
jUCMNav to test the round-trip. The structural checks to do:

1. **Variables panel** — should show a single `variant_id` variable
   typed as the `VariantId` enumeration.
2. **Scenarios panel** — should list one scenario per variant. Each
   scenario's description should show the partial-order expression
   and the truncated list of original case IDs.
3. **Run a scenario** — pick e.g. `v1` and run the traversal. The
   path should follow the OR-fork branch that variant 1 takes,
   driven by the `variant_id == "v1"` condition the synthesizer
   wrote.

If anything in jUCMNav looks off (variable not recognised, scenario
won't run, condition not evaluated), that's a structural-XMI bug to
report. The reference file used for cross-checking the output
structure is `aemsURN.jucm` (variables in `<ucmspec>`, attribute
order on `<initializations>` etc.).